# Inférence statistique

## Premiers tests de modélisation

In [36]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, r2_score


from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

In [37]:
df_train = pd.read_csv("../data_finale/train.csv")
df_val = pd.read_csv("../data_finale/val.csv")
df_test = pd.read_csv("../data_finale/test.csv")

In [38]:
colonne_cible = "market_value_in_eur"
colonne_joueur = "player"
colonne_team = "team"
colonne_nation = "nation"

df_train = df_train.dropna(subset=[colonne_cible])
df_val = df_val.dropna(subset=[colonne_cible])
df_test = df_test.dropna(subset=[colonne_cible])

# Séparation des features et de la variable cible
joueurs_test = df_test[colonne_joueur]

# On supprime les colonnes texte ET la cible pour l'entraînement
X_train = df_train.drop(columns=[colonne_cible, colonne_joueur, colonne_team, colonne_nation, "position",
                                 "market_value_in_eur_nor"])
y_train = df_train[colonne_cible]

X_val = df_val.drop(columns=[colonne_cible, colonne_joueur, colonne_team, colonne_nation, "position",
                                 "market_value_in_eur_nor"])
y_val = df_val[colonne_cible]

X_test = df_test.drop(columns=[colonne_cible, colonne_joueur, colonne_team, colonne_nation, "position",
                                 "market_value_in_eur_nor"])
y_test = df_test[colonne_cible]

# Définition des modèles
modeles = {
    "Random Forest": RandomForestRegressor(random_state=42, n_jobs=-1),
    "XGBoost": XGBRegressor(random_state=42, n_jobs=-1),
    "LightGBM": LGBMRegressor(random_state=42, verbose=-1, n_jobs=-1)
}

In [39]:
# Entraînement puis évaluation
resultats = {}

print("Début du benchmark des modèles d'arbres (Régression)...\n")

for nom, modele in modeles.items():
    print(f"Entraînement de {nom}...")
    
    # Entraînement sur le jeu de train uniquement
    modele.fit(X_train, y_train)
    
    # Prédictions
    preds_val = modele.predict(X_val)
    preds_test = modele.predict(X_test)
    
    mae_val = mean_absolute_error(y_val, preds_val)
    r2_val = r2_score(y_val, preds_val)
    
    mae_test = mean_absolute_error(y_test, preds_test)
    r2_test = r2_score(y_test, preds_test)
    
    # Sauvegarde pour le tableau final
    resultats[nom] = {
        "MAE Val": mae_val,
        "R² Val": r2_val,
        "MAE Test": mae_test,
        "R² Test": r2_test
    }
    
    print(f"   -> Validation | Erreur moyenne : {mae_val:,.0f} € | Score R² : {r2_val:.2%}")
    print(f"   -> Test Final | Erreur moyenne : {mae_test:,.0f} € | Score R² : {r2_test:.2%}\n")

# Comparaion des modèles
print("CLASSEMENT FINAL (Trié par la plus petite erreur sur test)")
tableau_resultats = []
for nom, metrics in resultats.items():
    tableau_resultats.append({
        "Modèle": nom,
        "Erreur moyenne Test (MAE)": f"{metrics['MAE Test']:,.0f} €",
        "Score R² Test": f"{metrics['R² Test']:.2%}"
    })

# En régression, le meilleur modèle est celui avec la MAE la plus basse
df_final = pd.DataFrame(tableau_resultats).sort_values(by="Erreur moyenne Test (MAE)", ascending=True)
print(df_final.to_string(index=False))

Début du benchmark des modèles d'arbres (Régression)...

Entraînement de Random Forest...
   -> Validation | Erreur moyenne : 5,059,658 € | Score R² : 71.50%
   -> Test Final | Erreur moyenne : 5,312,200 € | Score R² : 69.62%

Entraînement de XGBoost...
   -> Validation | Erreur moyenne : 5,034,851 € | Score R² : 73.60%
   -> Test Final | Erreur moyenne : 5,314,276 € | Score R² : 72.04%

Entraînement de LightGBM...
   -> Validation | Erreur moyenne : 4,891,153 € | Score R² : 72.90%
   -> Test Final | Erreur moyenne : 5,100,338 € | Score R² : 73.33%

CLASSEMENT FINAL (Trié par la plus petite erreur sur test)
       Modèle Erreur moyenne Test (MAE) Score R² Test
     LightGBM               5,100,338 €        73.33%
Random Forest               5,312,200 €        69.62%
      XGBoost               5,314,276 €        72.04%


## Auto-ML

In [4]:
%pip install pycaret

  Using cached pycaret-3.3.2-py3-none-any.whl.metadata (17 kB)
  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
  Using cached numpy-1.26.4.tar.gz (15.8 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'error'
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [21 lines of output]
      + c:\Users\LouisHarle\OneDrive - Quadratic\Bureau\Stage_VM\.venv\Scripts\python.exe C:\Users\LouisHarle\AppData\Local\Temp\pip-install-s44whf9p\numpy_ca2987ee25ce471d97522c328072d519\vendored-meson\meson\meson.py setup C:\Users\LouisHarle\AppData\Local\Temp\pip-install-s44whf9p\numpy_ca2987ee25ce471d97522c328072d519 C:\Users\LouisHarle\AppData\Local\Temp\pip-install-s44whf9p\numpy_ca2987ee25ce471d97522c328072d519\.mesonpy-mc2rhgai -Dbuildtype=release -Db_ndebug=if-release -Db_vscrt=md --native-file=C:\Users\LouisHarle\AppData\Local\Temp\pip-install-s44whf9p\numpy_ca2987ee25ce471d97522c328072d519\.mesonpy-mc2rhgai\meson-python-native-file.ini
      The Meson build system
      Version: 1.2.99
      Source dir: C:\Users\LouisHarle\AppData\Local\Temp\pip-install-s44whf9p\numpy_ca2987ee25ce471d97522c328072d519
      Build dir: C:\Users